# Compact Hugging Face Text Classification Demo
This notebook is designed for an academic presentation: simple, fast, and clear.

Use case: binary text classification with a small sampled dataset, then train/evaluate/save/infer.

## 1) Environment & Dependencies
Install required packages (run once).

In [ ]:
# If needed, uncomment and run:
# %pip install -q transformers datasets evaluate torch scikit-learn matplotlib accelerate

import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2) Load Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 3) Prepare Dataset (small / sample)
For speed, we use IMDB and sample a small subset. This is ideal for a classroom demo.

In [ ]:
raw = load_dataset('imdb')

# Small subset for quick training
train_ds = raw['train'].shuffle(seed=42).select(range(2000))
val_ds = raw['test'].shuffle(seed=42).select(range(500))
test_ds = raw['test'].shuffle(seed=123).select(range(500, 1000))

print('Train:', len(train_ds), 'Validation:', len(val_ds), 'Test:', len(test_ds))

## 4) Quick Exploratory Checks

In [ ]:
print('Example text snippet:', train_ds[0]['text'][:200], '...')

labels = [item['label'] for item in train_ds]
unique, counts = np.unique(labels, return_counts=True)
print('Label distribution:', dict(zip(unique, counts)))

lengths = [len(item['text'].split()) for item in train_ds.select(range(200))]
print('Avg token length (200 samples):', int(np.mean(lengths)))
print('Num classes:', len(set(labels)))

## 5) Preprocessing & Tokenization

In [ ]:
checkpoint = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128,
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)
test_tok = test_ds.map(tokenize_fn, batched=True)

cols = ['input_ids', 'attention_mask', 'label']
train_tok.set_format('torch', columns=cols)
val_tok.set_format('torch', columns=cols)
test_tok.set_format('torch', columns=cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization done.')

## 6) Create Model & Label Mapping (pretrained)

In [ ]:
id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
print('Model loaded.')

## 7) TrainingArguments & Trainer Setup

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average='binary')
    return {'accuracy': acc['accuracy'], 'f1': f1['f1']}

training_args = TrainingArguments(
    output_dir='./demo_model_ckpt',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print('Trainer ready.')

## 8) Train the Model

In [ ]:
# Start training (1 epoch for quick demo). Increase to 2-3 for better results.
train_result = trainer.train()
print('Training complete.')
print(train_result)

# Resume example if interrupted:
# trainer.train(resume_from_checkpoint=True)

## 9) Evaluate on Validation/Test Set

In [ ]:
val_metrics = trainer.evaluate(eval_dataset=val_tok)
test_metrics = trainer.evaluate(eval_dataset=test_tok)

print('Validation metrics:', val_metrics)
print('Test metrics:', test_metrics)

pred_output = trainer.predict(test_tok)
preds = np.argmax(pred_output.predictions, axis=-1)
true = pred_output.label_ids

print('\nClassification report:')
print(classification_report(true, preds, target_names=['NEGATIVE', 'POSITIVE']))

cm = confusion_matrix(true, preds)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NEGATIVE', 'POSITIVE']).plot(cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.show()

## 10) Save and Reload Model

In [ ]:
save_path = './saved_demo_model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print('Saved to', save_path)

reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_path)
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_path)
print('Reload successful.')

## 11) Inference Examples (interactive)

In [ ]:
import torch.nn.functional as F

def predict_text(text: str):
    inputs = reloaded_tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = reloaded_model(**inputs).logits
    probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    pred_id = int(np.argmax(probs))
    return {
        'text': text,
        'predicted_label': id2label[pred_id],
        'probabilities': {
            'NEGATIVE': float(probs[0]),
            'POSITIVE': float(probs[1]),
        }
    }

examples = [
    'This product is amazing, high quality and works perfectly.',
    'Terrible experience, very bad and disappointing.',
    'It is okay, not great but not bad either.'
]

for ex in examples:
    print(predict_text(ex))

# Interactive input
custom_text = 'I love this item, it is very useful and well made.'
print('\nCustom prediction:')
print(predict_text(custom_text))

## 12) (Optional) Simple Metric Plots

In [ ]:
history = trainer.state.log_history

train_loss_steps = [h for h in history if 'loss' in h and 'epoch' in h]
eval_steps = [h for h in history if 'eval_loss' in h and 'epoch' in h]

if train_loss_steps or eval_steps:
    plt.figure(figsize=(8, 4))
    if train_loss_steps:
        plt.plot([x['epoch'] for x in train_loss_steps], [x['loss'] for x in train_loss_steps], label='train_loss')
    if eval_steps:
        plt.plot([x['epoch'] for x in eval_steps], [x['eval_loss'] for x in eval_steps], label='eval_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.show()
else:
    print('No log history found yet. Train first.')